# Morphology Profile Prediction

In order to evaluate our ability to predict morphology from functional data we attempt to fit every feature in the (control-normalized) CellProfiler profiles using shallow models.

Since we have per-well profiles, we will compare two approaches:

- Fitting and predicting per-gene profiles, that are obtained by averaging the well profiles for the gene knockouts
- Fitting per-well profiles and predicting average per-gene profiles. In this evaluation we will also look at feature predictability ranked against feature variance across wells.

## Load profiles

In [3]:
import pandas as pd
from pathlib import Path

data_folder = Path('../data/cpg0016/version_2024-04-15')

# Load profiles
normalized_profiles = pd.read_parquet(data_folder/'normalized_profiles').set_index(['Metadata_Plate', 'Metadata_Well'])

In [10]:
# Get the list of knockouts
knockouts = normalized_profiles[normalized_profiles['Metadata_Perturbation'] == 'CRISPR-trt']['Metadata_Symbol'].drop_duplicates()
knockouts

Metadata_Plate  Metadata_Well
CP-CC9-R1-01    A03              ARHGEF7
                A04                 ST13
                A05                 GPHN
                A06                 EXT2
                A07                SNUPN
                                  ...   
CP-CC9-R2-20    O16              SMARCA4
CP-CC9-R2-21    L08                TGIF1
CP-CC9-R2-26    N09               LRSAM1
CP-CC9-R6-02    C09               INPP5B
CP-CC9-R7-02    J10               SPTLC1
Name: Metadata_Symbol, Length: 7975, dtype: object

In [35]:
# Should we select only knockouts first?
profile_columns = normalized_profiles.columns[~normalized_profiles.columns.str.startswith('Metadata')]

# Compute average gene profiles
profiles_groupby = normalized_profiles.groupby('Metadata_Symbol')[profile_columns]
gene_profiles = profiles_groupby.mean()

# Compute gene profile variances
gene_profile_sd = profiles_groupby.std()

## Build feature vectors
Next we collect our functional features

In [12]:
# Gene abundances
cell_line_df = pd.read_table('../../../data/cellular-localization/gene_abundances_U2OS.tsv')
cell_line_df

,Enesembl ID,Gene,RNA line ab,RNA type ab,protein type ab
0,ENSG00000000003,TSPAN6,23.4,NaN,NaN
1,ENSG00000000005,TNMD,0.0,NaN,NaN
2,ENSG00000000419,DPM1,97.8,NaN,NaN
3,ENSG00000000457,SCYL3,3.6,NaN,NaN
4,ENSG00000000460,C1orf112,16.1,NaN,NaN
...,...,...,...,...,...
20077,ENSG00000288677,AC105206.4,1.7,NaN,NaN
20078,ENSG00000288678,AL136115.4,0.0,NaN,NaN
20079,ENSG00000288681,AC136475.9,0.0,NaN,NaN
20080,ENSG00000288684,AL353671.1,0.7,NaN,NaN


In [13]:
# Undetected genes
undetected_df = pd.read_table('../../../data/cellular-localization/undetected_genes_U2OS.tsv')
undetected_df

,Enesembl ID,Gene
0,ENSG00000159455,LCE2B
1,ENSG00000141255,SPATA22
2,ENSG00000259303,IGHV2OR16-5
3,ENSG00000186930,KRTAP6-2
4,ENSG00000211967,IGHV3-53
...,...,...
4041,ENSG00000142973,CYP4B1
4042,ENSG00000188120,DAZ1
4043,ENSG00000111783,RFX4
4044,ENSG00000188730,VWC2


We will use the `RNA line ab` column and encode protein absence with a binary variable (absent=0, present=1):

In [28]:
abundance_df = cell_line_df[['Gene', 'RNA line ab']]
abundance_df.loc[:, 'Protein present'] = 1
abundance_df.loc[abundance_df['Gene'].isin(undetected_df['Gene']), 'Protein present'] = 0
abundance_df = abundance_df.set_index('Gene')
abundance_df

/tmp/ipykernel_442658/2026915113.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  abundance_df.loc[:, 'Protein present'] = 1


,RNA line ab,Protein present
Gene,,
TSPAN6,23.4,1
TNMD,0.0,0
DPM1,97.8,1
SCYL3,3.6,1
C1orf112,16.1,1
...,...,...
AC105206.4,1.7,1
AL136115.4,0.0,0
AC136475.9,0.0,0


In [15]:
# Subcellular localization data
sc_df = (
    pd
    .read_table(
        '../../../data/subcellular-localization/uniprot_reactome_hpa_merged.tsv',
        usecols=['gene_id', 'hpa_location']
    )
    .drop_duplicates()
    .pivot_table(index='gene_id', columns='hpa_location', aggfunc=lambda _: 1, fill_value=0)
)
sc_df

hpa_location,actin filaments:cytoplasm,aggresome:cytosol:cytoplasm,cell junctions:plasma membrane:endomembrane system,centriolar satellite:centrosome:cytoplasm,centrosome:cytoplasm,cleavage furrow:actin filaments:cytoplasm,cytokinetic bridge:microtubules:cytoplasm,cytoplasm,cytoplasmic bodies:cytosol:cytoplasm,cytosol:cytoplasm,...,nucleoli rim:nucleoli:nucleus,nucleoli:nucleus,nucleoplasm:nucleus,nucleus,peroxisomes:vesicles:endomembrane system,plasma membrane:endomembrane system,rods & rings:cytosol:cytoplasm,secreted proteins:secretory,secretory,vesicles:endomembrane system
gene_id,,,,,,,,,,,,,,,,,,,,,
1C,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1a,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3a,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,1,0,0,1,1
3b,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vpr,0,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,0,0,0,1,0
vpu,0,0,0,0,0,0,0,0,0,1,...,0,0,1,0,0,1,0,0,1,0
wnt11,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [16]:
# GO embeddings
go_embedding = pd.read_csv('../../../data/GO_Embeddings/go_embedding_64.csv', index_col=0)
go_embedding

,go_feature_0,go_feature_1,go_feature_2,go_feature_3,go_feature_4,go_feature_5,go_feature_6,go_feature_7,go_feature_8,go_feature_9,...,go_feature_54,go_feature_55,go_feature_56,go_feature_57,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63
A1BG,-0.367388,-0.413445,-0.404171,-0.004415,0.247503,-0.102804,0.157222,0.135817,0.089900,0.074568,...,0.292506,0.002101,0.263603,0.169054,0.063430,-0.171397,0.077323,0.051098,-0.082001,-0.036673
ABG,-0.158945,-0.490194,-0.403563,-0.029135,0.218117,-0.182563,0.094209,0.367191,0.190919,0.060625,...,0.354492,0.080943,0.412824,0.237152,0.164620,-0.067162,0.131368,-0.198871,-0.009133,-0.123394
GAB,-0.180069,-0.461517,-0.384918,0.124994,0.370956,-0.238049,-0.142643,0.268287,0.099368,-0.046293,...,0.083225,0.151852,0.395230,0.325642,0.009123,0.005238,0.014833,-0.086011,-0.110260,-0.070421
HYST2477,-0.264915,-0.487703,-0.230273,-0.033565,0.199071,-0.205979,0.118402,0.201433,0.175869,0.112664,...,0.110051,0.018307,0.547518,0.345197,0.265257,-0.011448,-0.016393,-0.227499,-0.241314,-0.233270
AGP-B,-0.503596,-0.640465,-0.509292,0.149607,0.393729,-0.390091,0.261283,0.243027,0.251378,-0.035333,...,0.491511,0.201053,0.626971,0.418596,0.004392,0.176189,0.066519,-0.039353,0.027906,-0.251306
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
BECN2,0.244011,-0.263258,-0.058161,0.086946,-0.049276,0.201113,-0.158159,-0.080029,0.105038,-0.029396,...,0.111178,0.110147,0.376339,0.228923,-0.087186,0.278534,-0.173626,-0.059988,0.162028,-0.135971
BECN1L1,0.232987,-0.238887,0.152314,0.086158,0.211937,0.196062,-0.173626,0.207144,0.054919,-0.048840,...,0.261120,-0.020350,0.431131,0.151029,0.037241,0.242878,-0.135145,-0.076569,0.097100,-0.015145
BECN1P1,0.203444,-0.165649,-0.144183,0.142063,0.079616,0.128623,-0.151379,-0.144399,0.001635,0.177796,...,0.199253,-0.054431,0.236613,-0.019560,-0.047433,0.272289,-0.071594,0.187233,-0.104395,0.001068
SULT1C3,0.061440,0.042919,-0.101767,0.142327,0.153482,0.057794,-0.162811,0.157489,0.196053,-0.061985,...,0.026004,-0.056475,0.208966,-0.208943,-0.094607,-0.046538,-0.100566,0.038614,-0.121547,0.020993


### Assembling the feature vectors for genes

In [17]:
normalized_profiles

Metadata_Source Metadata_JCP2022  \
Metadata_Plate Metadata_Well                                    
CP-CC9-R1-01   A02                 source_13   JCP2022_800002   
               A03                 source_13   JCP2022_800573   
               A04                 source_13   JCP2022_806794   
               A05                 source_13   JCP2022_802800   
               A06                 source_13   JCP2022_802216   
...                                      ...              ...   
CP-CC9-R8-02   P20                 source_13   JCP2022_801694   
               P21                 source_13   JCP2022_804281   
               P22                 source_13   JCP2022_800983   
               P23                 source_13   JCP2022_800001   
               P24                 source_13   JCP2022_046054   

                                          Metadata_Symbol  \
Metadata_Plate Metadata_Well                                
CP-CC9-R1-01   A02                          non-targeting   
               A03                                ARHGEF7   
               A04                                   ST13   
               A05                                   GPHN   
               A06                                   EXT2   
...                                                   ...   
CP-CC9-R8-02   P20                                DCLRE1A   
               P21                                  MS4A8   
               P22                                    CA7   
               P23                               no-guide   
               P24            KPBNHDGDUADAGP-UHFFFAOYSA-N   

                             Metadata_NCBI_Gene_ID Metadata_Perturbation  \
Metadata_Plate Metadata_Well                                               
CP-CC9-R1-01   A02                            None  CRISPR-non-targeting   
               A03                            8874            CRISPR-trt   
               A04                            6767            CRISPR-trt   
               A05                           10243            CRISPR-trt   
               A06                            2132            CRISPR-trt   
...                                            ...                   ...   
CP-CC9-R8-02   P20                            9937            CRISPR-trt   
               P21                           83661            CRISPR-trt   
               P22                             766            CRISPR-trt   
               P23                            None       CRISPR-no-guide   
               P24                            None          Compound-trt   

                              Cells_AreaShape_Area  \
Metadata_Plate Metadata_Well                         
CP-CC9-R1-01   A02                        2.300298   
               A03                        1.428099   
               A04                        5.046471   
               A05                        4.724336   
               A06                        5.282117   
...                                            ...   
CP-CC9-R8-02   P20                        4.961450   
               P21                        6.771890   
               P22                        5.684483   
               P23                        0.862481   
               P24                        8.697691   

                              Cells_AreaShape_BoundingBoxArea  \
Metadata_Plate Metadata_Well                                    
CP-CC9-R1-01   A02                                   2.341745   
               A03                                   1.455535   
               A04                                   5.163721   
               A05                                   4.665389   
               A06                                   5.154575   
...                                                       ...   
CP-CC9-R8-02   P20                                   5.206949   
               P21                                   7.169979   
               P22                             

In [33]:
gene_features = (
    go_embedding
        .merge(
            abundance_df,
            how='inner',
            left_index=True,
            right_index=True            
        )
        .merge(
            sc_df,
            how='inner',
            left_index=True,
            right_index=True
        )
)
gene_features.filter(knockouts.values)
gene_features

,go_feature_0,go_feature_1,go_feature_2,go_feature_3,go_feature_4,go_feature_5,go_feature_6,go_feature_7,go_feature_8,go_feature_9,...,nucleoli rim:nucleoli:nucleus,nucleoli:nucleus,nucleoplasm:nucleus,nucleus,peroxisomes:vesicles:endomembrane system,plasma membrane:endomembrane system,rods & rings:cytosol:cytoplasm,secreted proteins:secretory,secretory,vesicles:endomembrane system
A1BG,-0.367388,-0.413445,-0.404171,-0.004415,0.247503,-0.102804,0.157222,0.135817,0.089900,0.074568,...,0,0,0,0,0,0,0,1,1,0
A1CF,-0.374157,0.113069,-0.131485,-0.041938,0.248810,0.124938,-0.279050,-0.177576,-0.411285,-0.440202,...,0,0,1,1,0,0,0,0,0,0
A2M,0.276424,-0.109184,0.123967,-0.136654,-0.080143,0.040733,0.062247,0.087517,-0.191732,-0.037698,...,0,0,0,0,0,0,0,1,1,0
A2ML1,-0.295155,-0.216357,-0.508335,-0.260882,0.022338,0.226696,0.147074,0.231516,0.215579,0.252252,...,0,0,0,0,0,0,0,1,0,0
A3GALT2,0.350508,0.197309,0.149074,-0.393604,0.074340,-0.317506,-0.373907,-0.040924,0.226582,0.448294,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZYG11A,-0.373283,-0.040802,-0.141792,-0.079707,0.172872,-0.545426,0.155357,0.043712,-0.237321,0.160487,...,0,0,1,0,0,0,0,0,0,0
ZYG11B,-0.386697,0.045973,-0.168646,-0.183637,-0.126859,-0.749507,0.351075,-0.046246,-0.208723,0.262623,...,0,0,0,0,0,0,0,0,0,0
ZYX,-0.063201,0.010587,-0.127434,0.085080,0.049783,0.133588,0.149709,-0.026030,-0.165763,0.075462,...,0,0,0,1,0,1,0,0,0,0
ZZEF1,0.176582,0.022434,0.151153,0.247930,-0.133786,0.029006,0.274132,0.076891,-0.063100,0.082884,...,0,0,1,0,0,0,0,0,0,0


In [26]:
gene_features

,go_feature_0,go_feature_1,go_feature_2,go_feature_3,go_feature_4,go_feature_5,go_feature_6,go_feature_7,go_feature_8,go_feature_9,...,go_feature_57,go_feature_58,go_feature_59,go_feature_60,go_feature_61,go_feature_62,go_feature_63,Gene,RNA line ab,Protein present
5110,-0.367388,-0.413445,-0.404171,-0.004415,0.247503,-0.102804,0.157222,0.135817,0.089900,0.074568,...,0.169054,0.063430,-0.171397,0.077323,0.051098,-0.082001,-0.036673,A1BG,0.3,1
18252,-0.529263,-0.636665,-0.338881,-0.006108,0.348606,-0.447002,0.231994,0.396251,0.256772,-0.094459,...,0.447358,0.012635,-0.211489,0.052499,-0.152644,-0.303266,-0.241743,ORM2,0.1,1
12968,-0.067874,-0.448706,0.204147,0.300285,0.239869,0.266160,-0.068423,-0.134094,-0.190558,0.283061,...,0.272176,0.183957,-0.108499,0.164623,-0.358056,0.347152,-0.496522,SNTB1,1.1,1
13644,0.276424,-0.109184,0.123967,-0.136654,-0.080143,0.040733,0.062247,0.087517,-0.191732,-0.037698,...,0.219358,-0.083944,0.143957,-0.169459,-0.082821,-0.136685,0.475783,A2M,4.3,1
16426,0.474768,0.009743,-0.112181,-0.222603,-0.115097,-0.127075,0.200595,0.394848,-0.584848,0.145374,...,-0.097914,0.028863,0.254443,-0.227302,-0.279086,-0.239732,0.907282,SERPINA1,56.2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18266,-0.318409,-0.097855,0.120263,0.193515,0.145665,0.070527,0.219613,-0.074039,-0.096698,-0.063455,...,-0.316819,-0.095876,0.119833,-0.113023,-0.303185,0.363712,-0.071460,CT45A5,4.3,1
19391,0.067055,-0.261821,0.350655,0.068204,0.279183,0.652760,0.463589,0.043970,-0.227273,-0.105356,...,-0.634090,-0.306561,0.357268,0.175828,-0.297315,-0.159642,0.012886,CT45A10,14.2,1
18783,0.222549,0.061374,0.127283,-0.209418,-0.112109,-0.168855,-0.280916,-0.115899,-0.035165,0.306535,...,-0.185838,-0.166746,0.304659,0.059970,0.158506,0.165553,0.179465,TARM1,0.0,0
16162,0.244011,-0.263258,-0.058161,0.086946,-0.049276,0.201113,-0.158159,-0.080029,0.105038,-0.029396,...,0.228923,-0.087186,0.278534,-0.173626,-0.059988,0.162028,-0.135971,BECN2,0.0,0
